In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
# tensorflow.kreas 인스톨
# pip install tensorflow.keras
import numpy as np
import matplotlib.pyplot as plt
# matplotlib : 그래프 그리는 라이브러리

def mnist_deep_learning():
    """
    MNIST 손글씨 숫자를 인식하는 딥러닝 모델
    """

    # 1. 데이터 로드 및 전처리
    (x_train, y_train), (x_test, y_test) = mnist.load_data()
    # mnist : 손글씨 데이터셋
    # x_train : 훈련 데이터 (28x28 픽셀 이미지)
    # y_train : 훈련 데이터 레이블 (0~9)
    # x_test : 테스트 데이터 (28x28 픽셀 이미지)
    # y_test: 테스트 데이터 레이블 (0~9)

    # x_train_one 데이터 파일로 저장
    x_train_one = x_train[0]
    plt.imsave('x_train_one.png', x_train_one, cmap='gray')

    # 데이터 정규화 (0~255 -> 0~1)
    x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
    # reshape(-1, 28, 28, 1): 28x28 픽셀 이미지를 1채널로 변환
    # -1: 데이터 개수를 자동으로 계산
    # 28: 이미지 높이
    # 28: 이미지 너비
    # 1: 채널 수 (흑백 이미지이므로 1)
    # 1채널 (흑백): 각 픽셀이 1개 값
    # 픽셀값 = [0, 128, 255] # 검정, 회색, 흰색
    
    # # 3채널 (컬러): 각 픽셀이 3개 값
    # 픽셀값 = [
    # [255, 0, 0],    #빨강 (R = 255, G = theta, B = 0)
    # [0, 255, 0],    #초록 (R = 0, G = 255, B = 0)
    # [0, 0, 255]    #파랑 (R = theta, G = theta, B = 255)
    # ]
    # astype('float32') : 데이터 타입을 float32로 변환
    # / 255.0: 0~255 범위를 0~1 범위로 정규화

    print(x_train.shape)
    # (60000, 28, 28, 1): 60000개의 28x28 픽셀 이미지, 1채널
    # #60000개의 이미지가 있고, 각 이미지는 28x28 픽셀이며,
    # 각 픽셀은 1개의 채널(흑백) 값을 가지는 4차원 배열
    print(x_train[0])             # 첫 번째 이미지 (28, 28, 1)
    print(x_train[0][10])         # 첫 번째 이미지의 10번째 행 (28, 1)
    print(x_train[0] [10] [5])    # 첫 번째 이미지의 10행 5일 픽셀 (1,)
    print(x_train[0] [10][5][0])  # 해당 픽셀의 흑백 값

    X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

    #레이블 원-핫 인코딩
    y_train = to_categorical(y_train, 10)
    # to_categorical(y_train, 10): 레이블을 원-핫 인코딩으로 변환
    #10 : 카테고리 수 (0~9)
    # 값이 1이면, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0
    #값이 2이면, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0
    #값이 3이면, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0

    y_test = to_categorical(y_test, 10)

    # 2. 딥러닝 모델 구성 (CNN)
    # CNN: Convolutional Neural Network (컨볼루션 신경망)
    #
    # 왜 CNN을 사용하는가?
    # - 이미지는 공간적 정보가 중요 (픽셀들의 위치 관계)
    # - 일반 Dense 레이어는 공간 정보를 무시하고 모든 픽셀을 독립적으로 처리
    # -CNN은 이미지의 국소적 패턴(선, 모서리, 도형)을 효과적으로 학습
    #
    # CNN의 주요 구성 요소:
    # 1) 컨볼루션 레이어(Conv2D): 특징 추출 (필터로 이미지를 스캔)
    # 2) 폴링 레이어(MaxPooling2D): 크기 축소 + 중요한 정보만 유지
    # 3) 완전연결층(Dense): 최종 분류 결정

    model = Sequential([
        #첫 번째 컨볼루션 블록
        Conv2D (32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        # Conv2D 파라미터 상세 설명:
        #32 : 필터(커널) 개수 = 32가지 다른 특징을 동시에 찾는다
        #     (수직선, 수평선, 대각선, 곡선, 모서리 등)
        #(3,3): 필터 크기 = 3x3 픽셀 영역을 한번에 처리
        # activation='relu': ReLU 활성화 함수 (음수>0, 양수 그대로)
        # input_shape=(28, 28, 1) : 입력 형태(높이, 너비, 채널)
        #
        #동작 과정: 28x28 입력 → 3x3 필터로 스캔 26x26x32 출력
        # 크기가 줄어드는 이유 (Valid Convolution):
        # - 3x3 필터 전체가 이미지 안에 들어가야 함 (경계 벗어나기 불가)
        # - 필터 중심 위치: 최소 (1,1) ~ 최대 (26, 26) = 26 * 26 가능 위치
        # 가장자리 1픽셀씩은 필터 중심이 될 수 없음

        MaxPooling2D((2, 2)),
        # MaxPooling2D 상세 설명:
        # (2, 2): 2x2 영역에서 최대값만 선택하여 크기를 절반으로 축소
        # 효과: 26x26x32 13x13x32 (계산량 감소, 위치 불변성 증가)
        # 예시: [[1,3], [2,4]] → 4 (최대값만 선택)

        # 두 번째 컨볼루션 블록
        Conv2D(64, (3, 3), activation='relu'),
        # 64개 필터 사용 (더 많은 고수준 특징 추출)
        # 입력: 13x13x32 → 출력: 11x11x64
        # 특징의 계층화: 1층(선, 모서리) → 2층(도형, 패턴) → 3층(복잡한 형태)
        MaxPooling2D((2, 2)),
        # 11x11x64 + 5x5x64 (더욱 축소)
        #세 번째 컨볼루션 블록
        Conv2D(64, (3, 3), activation='relu'),
        # 64개 필터로 더욱 복잡한 특징 추출
        # 입력: 5x5x64 → 출력: 3x3x64
        # 이 단계에서는 숫자의 전체 형태적 특징을 학습

        # 완전 연결층(분류를 위한 최종 단계)
        Flatten(),
        # Flatten(): 3D 특징맵을 1D 벡터로 변환
        # 3x3x64 = 576개 특징 576차원 벡터로 평탄화
        # CNN에서 추출한 공간적 특징들을 분류기에 전달하기 위한 준비
        Dense (64, activation='relu'),
        # Dense(뉴런수, 활성화함수) : 완전연결층 (모든 입력이 모든 출력에 연결)
        # 64 : 은닉층 뉴런 개수 (576개 특징을 64개로 압축 요약)
        # 'relu': ReLU 활성화 함수로 비선형성 추가
        # 역할: CNN이 추출한 특징들을 조합하여 더 추상적 표현 학습
        Dropout(0.5),
        # Dropout(비율) : 과적합 방지 기법
        # 0.5 : 훈련시 50% 뉴런을 무작위로 비활성화
        # 효과: 모델이 특정 뉴런에 과도하게 의존하는 것을 방지

        Dense(10, activation='softmax')
        #최종 출력층 : 10개 클래스(0~9 숫자) 분류
        #10: 출력 뉴런 개수 = 클래스 개수
        #'softmax': 확률 분포로 변환 (모든 출력의 합 = 1)
        #출력 예시: [0.1, 0.8, 0.05, 0.02, ...] → 숫자 '1'일 확률이 80%
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    #4. 모델 훈련 (학습 실행)
    print("모델 훈련을 시작합니다...")
    print(" 훈련 과정: CNN이 60,000개 이미지에서 패턴을 학습합니다...")
    history = model.fit(
        x_train, y_train,
        batch_size=128,
        epochs=5,
        validation_data=(x_test, y_test),
        verbose=1
    )
    
    #5. 모델 평가
    test_loss, test_accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"\n최종 테스트 정확도: {test_accuracy:.4f}")
    print(f"MNIST 벤치마크: 일반적으로 99% 이상 달성 가능")
    #6. 예측 예시 (실제 사용 시뮬레이션)
    sample_idx = np.random.randint(0, len(X_test), 5)
    predictions = model.predict(X_test[sample_idx])
    # predictions : 각 샘플에 대한 10개 클래스의 확률 분포
    #           예 : [[0.05, 0.85, 0.02, ...], [...], ...]

    predictions = model.predict(x_test[sample_idx])
    # predictions : 각 샘플에 대한 10개 클래스의 확률 분포
    #           예 : [[0.05, 0.85, 0.02, ...], [...], ...]
    
    print("\n실제 예측 테스트:")
    for i, idx in enumerate(sample_idx):
        predicted_digit = np.argmax(predictions[i])
        # np.argmax() : 확률이 가장 높은 클래스 선택
        #예: [0.05, 0.85, 0.02, ...] → 인덱스 1 반환 (85%가 최고)
        actual_digit = np.argmax(y_test[idx])
        #원-핫 인코딩에서 실제 숫자 추출
        #예: [0, 1, 0, 0, 0, 0, 0, 0, 0, 0] → 1

        confidence = np.max(predictions[i]) * 100
        # 최고 확률을 퍼센트로 변환 (모델의 확신도)

        result = "/" if predicted_digit == actual_digit else "X"
        print(f"샘플 {i+1}: 예측={predicted_digit}, 실제={actual_digit}, 신뢰도={confidence:.1f}%")

    print("\nCNN 모델 완성! 손글씨 숫자를 높은 정확도로 인식할 수 있습니다.")

    return model, history

#실행
model, history = mnist_deep_learning()

SyntaxError: invalid syntax (1097859809.py, line 126)